In [1]:
%cd ../../

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [17]:
import re

import numpy as np
import pandas as pd
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

# Load data

## fact 'waste'

In [3]:
path = "data/processed/waste.parquet"

waste = pl.read_parquet(path)
waste.head()

id,date,restaurant,waste,src
str,date,i64,f32,str
"""2024-11-01|3""",2024-11-01,3,19.0,"""Data Physicum 01112024-3103202…"
"""2024-11-04|3""",2024-11-04,3,21.0,"""Data Physicum 01112024-3103202…"
"""2024-11-05|3""",2024-11-05,3,15.0,"""Data Physicum 01112024-3103202…"
"""2024-11-06|3""",2024-11-06,3,15.0,"""Data Physicum 01112024-3103202…"
"""2024-11-07|3""",2024-11-07,3,15.0,"""Data Physicum 01112024-3103202…"


## fact 'pos'

In [4]:
path = "data/processed/pos.xlsx"

pos = pl.read_excel(path)
pos.head()

id,restaurant,meal_id,datetime,pcs,src
i64,i64,i64,datetime[ms],i64,str
0,1,55,2023-01-02 10:31:00,1,"""Sold lunches"""
1,1,8,2023-01-02 10:32:00,1,"""Sold lunches"""
2,1,55,2023-01-02 10:32:00,1,"""Sold lunches"""
3,1,8,2023-01-02 10:35:00,1,"""Sold lunches"""
4,1,55,2023-01-02 10:36:00,2,"""Sold lunches"""


## dim 'meals'

In [5]:
path = "data/processed/dim_meals.parquet"

dim_meals = (
    pl.read_parquet(path)
)

dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


# Extract biowaste info for each meal from historical POS data

In [6]:
waste_per_meal = (
    dim_meals
    .select(
        pl.col('id').alias('meal_id'),
        pl.lit(0.).alias('waste'),
        pl.Series([embd for embd in np.eye(len(dim_meals))]).alias('embd')
    )
    .to_pandas()
)

waste_per_meal.head()

,meal_id,waste,embd
0,0,0.0,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,1,0.0,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,2,0.0,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,3,0.0,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,4,0.0,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."


In [7]:
pos_accum = (
    pos
    .with_columns(pl.col('datetime').dt.date().alias('date'))
    .group_by('date', 'restaurant', 'meal_id')
    .agg(pl.col('pcs').sum())
    .to_pandas()
)



In [8]:
waste_merged = (
    pos_accum
    .merge(waste_per_meal, on='meal_id', how='left')
)
waste_merged['embd'] = waste_merged['embd'] * waste_merged['pcs']

waste_merged.head()

,date,restaurant,meal_id,pcs,waste,embd
0,2025-02-06,3,280,3,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,2023-12-08,2,44,1,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,2025-02-10,2,34,186,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,2023-10-20,3,352,11,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,2023-02-14,3,226,1,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [11]:
waste_byday = (
    waste_merged
    .drop(columns='waste')
    .groupby(['date', 'restaurant'])['embd']
    .sum()
    .reset_index()
    .merge(waste.to_pandas(), on=['date', 'restaurant'], how='left')
    [['date', 'restaurant', 'embd', 'waste']]
)

waste_byday.head()

,date,restaurant,embd,waste
0,2023-01-02,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 78.0,...",17.900000
1,2023-01-03,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 52.0,...",21.200001
2,2023-01-04,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",15.250000
3,2023-01-05,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",21.799999
4,2023-01-09,1,"[0.0, 0.0, 0.0, 0.0, 8.0, 0.0, 0.0, 0.0, 98.0,...",14.650000


In [12]:
waste_byday_train = waste_byday[waste_byday['date'] <= "2025-01-01"]

X = np.stack(waste_byday_train['embd'].to_numpy())
y = waste_byday_train['waste']

model = LinearRegression(positive=True, fit_intercept=False)
model.fit(X, y)


waste_per_meal['waste'] = model.coef_.copy()
waste_per_meal.drop(columns='embd', inplace=True)
waste_per_meal.head()

,meal_id,waste
0,0,0.000000
1,1,0.078735
2,2,0.051191
3,3,0.000000
4,4,0.021868


# Test

Calculate MAPE for whole-day waste for whole restaurant starting from '2025-01-01'

In [13]:
DATE_FROM = pl.lit('2025-01-01').str.to_date()
EPS = 1e-6

waste_pred = (
    pos
    .select(
        'restaurant',
        pl.col('datetime').dt.date().alias('date'),
        'meal_id',
        'pcs'
    )
    .filter(pl.col('date') >= DATE_FROM)
    
    # Add waste_info
    .join(pl.from_pandas(waste_per_meal), on='meal_id', how='left')
    .with_columns(((pl.col('waste') + EPS) * pl.col('pcs')).alias('waste'))


    # Accumulate total waste per restaurat per day
    .group_by('restaurant', 'date')
    .agg(pl.col('waste').sum().alias('waste_pred'))
)

waste_pred.head()

restaurant,date,waste_pred
i64,date,f64
4,2025-03-10,22.254456
4,2025-02-19,16.529168
4,2025-02-04,12.327783
2,2025-02-11,55.663448
3,2025-02-12,19.939003


In [22]:
(
    waste
    .filter(pl.col('date') >= DATE_FROM)
    .select('restaurant', 'date', 'waste')

    .join(waste_pred, on=['restaurant', 'date'], how='left')

    .filter(
        (1 == 1)
        & (pl.col('waste') > EPS)
        # (pl.col('waste_pred').is_null())
    )

    # Calculate MAPE
    .with_columns(
        ((pl.col('waste') - pl.col('waste_pred')).abs()/pl.col('waste')).alias('diff')
    )
    .group_by('restaurant')
    .agg(pl.col('diff').mean().alias('mape'))
    
)

# df_total.head()

restaurant,mape
i64,f64
2,0.560938
4,0.393645
3,0.356371
1,0.518531
